# Grandes modelos de lenguaje (LLMs)

En este notebook, exploraremos el uso de grandes modelos de lenguaje (LLMs) para tareas de procesamiento de lenguaje natural, con hincapié en aplicaciones clínicas y de salud. Cubriremos cómo conectarse a APIs de LLMs, enviar consultas, procesar respuestas y algunos ejemplos de prompting y buenas prácticas.

In [1]:
import requests
from openai import OpenAI
from pydantic import BaseModel
import json
import datasets
from sklearn.metrics import classification_report

## Autenticación

Para interactuar con un LLM, primero debemos autenticar nuestra aplicación. Esto generalmente implica obtener una clave API de un proveedor de LLM. Asegúrate de tener tu clave API a mano.

In [ ]:
API_KEY = ""

## API de OpenAI

A través de consultas HTTP a la API de OpenAI, podemos enviar solicitudes y recibir respuestas de un modelo de lenguaje. Aquí hay un ejemplo básico de cómo hacerlo en Python.

In [3]:
url = "https://models.villena.cl/v1/responses"
headers = {
    "Content-Type": "application/json",
}
headers["Authorization"] = f"Bearer {API_KEY}"
data = {
    "model": "openai/gpt-5-nano",
    "input": "Define qué es el procesamiento de lenguaje natural clínico.",
}
response = requests.post(url, headers=headers, json=data)
response.json()

{'id': 'resp_bGl0ZWxsbTpjdXN0b21fbGxtX3Byb3ZpZGVyOm9wZW5haTttb2RlbF9pZDo5ZjU1MDFkMC1lM2Y0LTQzNzEtOWNlZS0xM2JhY2ZlNDc5MzI7cmVzcG9uc2VfaWQ6cmVzcF82OGQ2ZmE5MmYxYTA4MTkzYTMxMzM3Y2M3N2E0YTg5MzBjZDBjMzlmODY2NWY1YTg=',
 'created_at': 1758919315,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'gpt-5-nano-2025-08-07',
 'object': 'response',
 'output': [{'id': 'rs_68d6fa941b988193ba81d4cc5445a9470cd0c39f8665f5a8',
   'summary': [],
   'type': 'reasoning',
   'content': None,
   'encrypted_content': None,
   'status': None},
  {'id': 'msg_68d6fa9c1cb881939bf19a5edf9efd420cd0c39f8665f5a8',
   'content': [{'annotations': [],
     'text': 'El procesamiento de lenguaje natural clínico es un subcampo del procesamiento de lenguaje natural que se ocupa del análisis de textos médicos y sanitarios para extraer y aprovechar información clínica no estructurada (por ejemplo, notas de evolución, informes, historias clínicas, informes de laboratorio) con el fin d

## Uso de la biblioteca `openai`

En vez de hacer solicitudes HTTP manualmente, podemos usar la biblioteca `openai` para simplificar el proceso. Asegúrate de instalarla primero:


In [4]:
client = OpenAI(
    api_key=API_KEY,
    base_url="https://models.villena.cl",
)

###  Uso básico

Podemos usar la biblioteca `openai` para enviar solicitudes a la API de OpenAI. Aquí hay un ejemplo básico de cómo hacerlo

In [5]:
response = client.responses.create(
    mode    instructions="Eres un experto en procesamiento de lenguaje natural clínico. Responde a la pregunta de manera clara y concisa.",
    input="¿Qué debo tomar en cuenta para desarrollar una función de preprocesamiento?",
)

print(response.output_text)

Para desarrollar una función de preprocesamiento en NLP clínico, considera estos aspectos clave:

1) Propósito y alcance
- Define qué tasks downstream alimentará (NER, clasificación, extracción de conceptos, etc.).
- Determina el nivel de detalle deseado (texto limpio, tokens, lemas, entidades normalizadas, etc.).

2) Origen y naturaleza de los datos
- Nota clínica, transcripciones, informes quirúrgicos, etc. Cada tipo tiene jerga, estructuras y rúbricas distintas.
- Idioma y variaciones regionales (abreviaturas, errores de transcripción, jerga).

3) Privacidad y seguridad
- Desidentificación/anonimización de PHI (nombre, fechas, direcciones, números de teléfono).
- Cumplimiento legal (HIPAA, GDPR) y entorno seguro (entornos aislados, control de acceso).
- Minimización de datos y trazabilidad adecuada de procesamientos.

4) Lenguaje clínico y diccionarios
- Abreviaturas, sinónimos y jerga clínica; necesidad de expandir abreviaturas (p. ej., “ICU” → Unidad de Cuidados Intensivos).
- Nor

También podemos usar imágenes como parte de nuestras consultas. A continuación, se muestra un ejemplo de cómo enviar una imagen junto con un mensaje de texto.


In [6]:
prompt = "¿Qué enfermedad es probable que tenga el paciente?"
img_url = "https://patoral.umayor.cl/canmucor/ca_leng_mb1.jpg"

response = client.responses.create(
    model="openai/gpt-4o",
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {"type": "input_image", "image_url": f"{img_url}"},
            ],
        }
    ],
)

print(response.output_text)

Lo siento, no puedo ayudar con la identificación de enfermedades o diagnósticos médicos a partir de imágenes. Es importante que consultes a un profesional de la salud para obtener una evaluación adecuada y precisa.


### Mensajes y roles

Los mensajes en la API de OpenAI tienen roles que indican quién está hablando. Los roles comunes son `system`, `user` y `assistant`. Aquí hay un ejemplo de cómo estructurar un mensaje

In [7]:
response = client.responses.create(
    model="openai/gpt-4o",
    input=[
        {"role": "developer", "content": "Eres un experto en salud digital."},
        {"role": "user", "content": "Qué es lo más importante al implementar un proyecto de informática médica?"}
    ]
)
print(response.output_text)


Al implementar un proyecto de informática médica, hay varios elementos clave que deben considerarse para asegurar su éxito:

1. **Interoperabilidad**: Asegurarse de que los sistemas puedan comunicarse y compartir datos entre sí de manera eficiente y segura.

2. **Seguridad y privacidad**: Implementar medidas robustas para proteger la información de salud del paciente, cumpliendo con regulaciones como la HIPAA en EE.UU.

3. **Participación de los usuarios**: Involucrar a médicos, enfermeros y otros profesionales de la salud en el diseño y desarrollo para que el sistema sea fácil de usar y satisfaga sus necesidades.

4. **Capacitación adecuada**: Ofrecer formación para que el personal de salud se sienta cómodo utilizando las nuevas tecnologías.

5. **Evaluación continua**: Monitorear y evaluar el rendimiento del sistema de manera continua para identificar áreas de mejora y adaptarse a las necesidades cambiantes.

6. **Integración con procesos existentes**: Asegurar que el nuevo sistema s

In [9]:
developer_message = """
# Identidad

Eres un priorizador de la lista de espera chilena y deben priorizar pacientes en las categorías Urgente o Rutina.

# Instrucciones

* Solo responde con una palabra: "Urgente" o "Rutina".
"""

response = client.responses.create(
    model="openai/gpt-5-nano",
    input=[
        {"role": "developer", "content": developer_message},
        {"role": "user", "content": "El paciente presenta un dolor muy leve en el dedo meñique izquierdo."}
    ]
)
print(response.output_text)


Rutina


### Adición de ejemplos

Para mejorar la calidad de las respuestas, podemos proporcionar ejemplos de preguntas y respuestas esperadas. Esto ayuda al modelo a entender mejor el contexto y las expectativas.

In [14]:
developer_message = """
# Identidad

Eres un priorizador de la lista de espera chilena y deben priorizar pacientes en las categorías Urgente o Rutina.

# Instrucciones

* Solo responde con una palabra: "Urgente" o "Rutina".

# Ejemplos

<interconsulta>
El paciente presenta pérdida de peso de 10 kg en los últimos 3 meses y tiene antecedentes de cáncer de colon.
</interconsulta>
<assistant_response">
Urgente
</assistant_response>
<interconsulta>
El paciente presenta un dolor muy leve en el dedo meñique izquierdo.
</interconsulta>
<assistant_response">
Rutina
</assistant_response>
<interconsulta>
El paciente tiene un dolor intenso en el pecho y dificultad para respirar.
</interconsulta>
<assistant_response">
Urgente
</assistant_response>
"""

response = client.responses.create(
    model="openai/gpt-5-nano",
    input=[
        {"role": "developer", "content": developer_message},
        {"role": "user", "content": "El paciente tiene un dolor intenso en la zona parietal izquierda."}
    ]
)
print(response.output_text)


Urgente


### Respuesta estructurada

Para obtener respuestas más estructuradas, podemos usar el parámetro `text_format` en la solicitud. Esto nos permite especificar el formato de la respuesta esperada.

In [15]:
# Definir el esquema de salida estructurada
class InterconsultaPrioridad(BaseModel):
    prioridad: str  # "Urgente" o "Rutina"

# Usar el método parse para obtener una respuesta estructurada
response = client.responses.parse(
    model="openai/gpt-5-nano",
    input=[
        {"role": "system", "content": "Eres un priorizador de la lista de espera chilena. Responde solo con la prioridad del paciente ('Urgente' o 'Rutina')."},
        {"role": "user", "content": "El paciente tiene fiebre alta y dificultad respiratoria."}
    ],
    text_format=InterconsultaPrioridad,
)

print(response.output_parsed.prioridad)

Urgente


En algunos casos, es fundamental que la respuesta del modelo siga una estructura específica y validable, especialmente para integraciones clínicas o flujos automatizados. La API de OpenAI permite definir un esquema JSON (JSON Schema) que el modelo debe seguir estrictamente en su respuesta. A continuación, se muestra cómo definir un esquema para priorización de interconsultas y cómo solicitar al modelo que responda usando exactamente ese formato estructurado.

In [17]:
schema = {
    "type": "object",
    "properties": {
        "prioridad": {
            "type": "string",
            "enum": ["Urgente", "Rutina"],
            "description": "Prioridad del paciente según la interconsulta"
        },
        "descripcion": {
            "type": "string",
            "description": "Descripción adicional de la interconsulta"
        }
    },
    "required": ["prioridad", "descripcion"],
    "additionalProperties": False
}

response = client.responses.create(
    model="openai/gpt-5-nano",
    input=[
        {"role": "system", "content": "Eres un priorizador de la lista de espera chilena. Responde solo con la prioridad del paciente ('Urgente' o 'Rutina')."},
        {"role": "user", "content": "El paciente refiere dolor abdominal leve desde hace 2 semanas."}
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "prioridad_interconsulta",
            "schema": schema,
            "strict": True
        }
    }
)

json.loads(response.output_text)

{'prioridad': 'Rutina',
 'descripcion': 'Dolor abdominal leve, sin signos de alarma descritos; dolor de 2 semanas. Sin indicios de urgencia.'}

### Chain of Thought (Cadena de Pensamiento)

El prompting tipo "chain of thought" (cadena de pensamiento) le indica al modelo que razone paso a paso antes de dar una respuesta final. Esto es útil para tareas complejas donde se requiere justificar o explicar el razonamiento detrás de la decisión.

A continuación, se muestra un ejemplo donde se le pide al modelo que explique su razonamiento antes de priorizar la interconsulta


In [18]:
cot_prompt = """
Eres un priorizador de la lista de espera chilena. 
Primero, analiza los síntomas del paciente paso a paso y explica tu razonamiento. 
Luego, responde con la prioridad final: "Urgente" o "Rutina".

Paciente: El paciente presenta fiebre alta, tos persistente y dificultad respiratoria.
"""

response = client.responses.create(
    model="openai/gpt-5-nano",
    input=[
        {"role": "system", "content": cot_prompt}
    ]
)

print(response.output_text)

Análisis paso a paso (razonamiento):

- Paso 1: Identificación de los síntomas principales. El paciente tiene fiebre alta, tos persistente y dificultad respiratoria. Estos síntomas pueden indicar una infección respiratoria aguda con posible compromiso pulmonar.

- Paso 2: Evaluación de gravedad. La dificultad respiratoria es un signo de alarma que sugiere posible neumonía, bronquitis severa u otra infección respiratoria que puede progresar rápido. La fiebre alta refuerza la posibilidad de infección significativa.

- Paso 3: Posibles etiologías a considerar. Neumonía adquirida en la comunidad, influenza, COVID-19 u otra infección respiratoria. Sin datos de signos vitales (p. ej., saturación de oxígeno, frecuencia cardíaca) es difícil precisar la severidad exacta, pero la tríada de fiebre + tos + disnea ya sugiere necesidad de evaluación clínica rápida.

- Paso 4: Factores de riesgo y evolución. La severidad puede agravarse de forma rápida, especialmente en adultos mayores, personas con 

## Clasificador utilizando LLMs

Los LLMs también pueden ser utilizados como clasificadores para tareas específicas, como la clasificación de interconsultas médicas. A continuación, se muestra un ejemplo de cómo implementar un clasificador utilizando un LLM.

In [19]:
spanish_diagnostics = datasets.load_dataset('fvillena/spanish_diagnostics')

In [20]:
test = spanish_diagnostics['test'].select(range(100))

In [25]:
def classifier(text):
    schema = {
        "type": "object",
        "properties": {
            "tipo": {
                "type": "string",
                "enum": ["dental", "no_dental"],
                "description": "Tipo de diagnóstico: 'dental' cuando se debe enviar la interconsulta a una especialidad dental o 'no_dental' cuando no se debe enviar a una especialidad dental"
            }
        },
        "required": ["tipo"],
        "additionalProperties": False
    }
    response = client.responses.create(
        model="openai/gpt-5-nano",
        input=[
            {"role": "system", "content": "Eres un clasificador de diagnósticos médicos."},
            {"role": "user", "content": text}
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "diagnostico_clasificacion",
                "schema": schema,
                "strict": True
            }
        }
    )
    return json.loads(response.output_text)["tipo"]

In [26]:
classifier("caries en el diente 12 y 13, dolor leve al masticar")

'dental'

In [ ]:
predicted = [classifier(item['text']) for item in test]

In [ ]:
print(classification_report(["dental" if item['label'] == 1 else "no_dental" for item in test], predicted))

              precision    recall  f1-score   support

      dental       0.93      0.98      0.96        44
   no_dental       0.98      0.95      0.96        56

    accuracy                           0.96       100
   macro avg       0.96      0.96      0.96       100
weighted avg       0.96      0.96      0.96       100



In [ ]:
def classifier_few_shot(text):
    schema = {
        "type": "object",
        "properties": {
            "tipo": {
                "type": "string",
                "enum": ["dental", "no_dental"],
                "description": "Tipo de diagnóstico: 'dental' cuando se debe enviar la interconsulta a una especialidad dental o 'no_dental' cuando no se debe enviar a una especialidad dental"
            }
        },
        "required": ["tipo"],
        "additionalProperties": False
    }
    response = client.responses.create(
        model="openai/gpt-5-nano",
        input=[
            {"role": "system", "content": "Eres un clasificador de diagnósticos médicos."},
            {"role": "user", "content": "paciente con dolor en el diente 12 y 13"},
            {"role": "assistant", "content": '{"tipo":"dental"}'},
            {"role": "user", "content": "paciente con dolor en la rodilla derecha"},
            {"role": "assistant", "content": '{"tipo":"no_dental"}'},
            {"role": "user", "content": text}
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "diagnostico_clasificacion",
                "schema": schema,
                "strict": True
            }
        }
    )
    return json.loads(response.output[0].content[0].text)["tipo"]

In [ ]:
classifier_few_shot("El paciente presenta dolor en el diente 12 y 13, con sensibilidad al frío y al calor.")

'dental'

In [ ]:
predicted = [classifier_few_shot(item['text']) for item in test]

In [ ]:
print(classification_report(["dental" if item['label'] == 1 else "no_dental" for item in test], predicted))

              precision    recall  f1-score   support

      dental       0.94      1.00      0.97        44
   no_dental       1.00      0.95      0.97        56

    accuracy                           0.97       100
   macro avg       0.97      0.97      0.97       100
weighted avg       0.97      0.97      0.97       100

